# T12 — 보조기·의지·웨어러블 로봇

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait-theory/note-T12.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait-theory/note-T12.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 본문

### 5. phase variable: 실시간으로 "주기의 몇 %"를 안다는 것

아래 코드는 두 방식이 고조파에 어떻게 반응하는지 가상 허벅지 각도로 비교한다. 순수 코사인에서는 둘 다 정확하지만, 2차 고조파를 섞으면 미분은 고조파를 키우고 적분은 줄이므로 각속도 방식의 위상 오차가 더 크다. 이 코드는 기록 전체 평균을 빼고 참 heel strike에서 적분을 되돌리므로 **오프라인 비교**다. 실시간에서는 정규화 범위와 heel strike도 추정해야 한다.

In [ ]:
import numpy as np

fs = 200                                   # 가상 허벅지 각도, 200 Hz, stride 1.1 s를 5번
T = 1.1
t = np.arange(0, 5 * T, 1 / fs)
u = 2 * np.pi * t / T                      # 참 위상(0~2π가 한 stride)
true = (u % (2 * np.pi)) / (2 * np.pi)     # 참 phase 0~1

def thigh(u, h2):                          # 코사인 + 2차 고조파(h2 = 0이면 순수 코사인)
    return 20 * np.cos(u) + h2 * 20 * np.cos(2 * u + 0.6) + 10

def err(est):                              # 0/1 경계를 고려한 위상 오차의 최대값(주기의 %)
    d = (est - true + 0.5) % 1 - 0.5
    return 100 * np.abs(d[t > T]).max()    # 첫 stride는 적분 초기화 구간이라 뺀다

for h2 in [0.0, 0.3]:
    q = thigh(u, h2)
    qc = q - q.mean()                      # 오프라인: 기록 전체 평균을 빼 중심을 맞춤
    # (가) [RM] 13장식 각도–각속도 평면
    v = np.gradient(qc, t)
    pa = np.arctan2(-(v / np.abs(v).max()), qc / np.abs(qc).max())   # 부호를 바꿔 시간에 따라 증가하게
    est_a = (pa % (2 * np.pi)) / (2 * np.pi)
    # (나) [R] 24장식 각도–적분 평면: 적분을 stride마다(참 위상 0에서) 0으로 되돌린다
    integ = np.zeros_like(qc)
    for i in range(1, len(qc)):
        integ[i] = 0 if true[i] < true[i - 1] else integ[i - 1] + qc[i] / fs
    integ -= integ.mean()
    k = np.ptp(qc) / np.ptp(integ)
    pb = np.arctan2(k * integ, qc)
    est_b = (pb % (2 * np.pi)) / (2 * np.pi)
    print(f"2차 고조파 {h2:.1f}: 최대 위상 오차 각속도 방식 {err(est_a):4.1f}% · 적분 방식 {err(est_b):4.1f}%")